In [ ]:
import sys,os
import numpy             as np
import matplotlib.pyplot as plt
import pandas            as pd
import seaborn           as sb

from itertools import product
from copy      import deepcopy
from time      import time
from tqdm      import tqdm

import warnings
warnings.filterwarnings('ignore')

from cobaya.run   import run

from source_code.likelihood import LSSlike

import matplotlib
from matplotlib import rc
rc('text', usetex=True)
rc('font', family='serif')
matplotlib.rcParams.update({'font.size': 18})

red    = '#8e001c'
yellow = '#ffb302'

sidelegend = {'bbox_to_anchor': (1.04,0.5), 
              'loc': "center left",
              'frameon': False}

In [ ]:
param_grids = {
    'H0': np.linspace(66.5, 67.5, 20),
    #'ombh2': np.linspace(0.01, 0.03, 20),  
    #'omch2': np.linspace(0.05, 0.2, 20),
}

In [ ]:
data_path = {'LSS': './mock_data/LCDM_test_gal',
             'LSS_GWC': './mock_data/LCDM_test_galGWC',
             'LSS_GWL': './mock_data/LCDM_test_galGWWL',
             'LSS_GWs': './mock_data/LCDM_test_galGWs',
             }

In [ ]:


info = {'sampler': {'mcmc': {'max_tries':100000}},
                             #'covmat': 'LCDM_covmat_3x2pt.covmat'}},
        'likelihood': {'LSS': {'external': LSSlike,
                               #'data_path': './mock_data/LCDM_test_gal',
                               'debug_mode': False,
                               'use_noiseless_cls': True,
                               'settings': { 'case': 'simple',
                                            'calculation': 'CAMB',
                                            'extra':{},
                                            'camb_path': {}},
                                            }}}


In [ ]:
open_params = { 'H0': {'latex': 'H_0',
                        'prior': {'max': 100.0, 'min': 40.0},
                        'proposal': 0.5,
                        'ref': {'dist': 'norm', 'loc': 67.0, 'scale': 1.0}},
                'ombh2': {'latex': '\Omega_\mathrm{b} h^2',
                        'prior': {'max': 0.1, 'min': 0.005},
                        'proposal': 0.0001,
                        'ref': {'dist': 'norm', 'loc': 0.0222, 'scale': 0.0001}},
                'omch2': {'latex':'\Omega_\mathrm{c} h^2',
                        'prior': {'max': 0.99, 'min': 0.002},
                        'proposal': 0.0005,
                        'ref': {'dist': 'norm', 'loc': 0.12, 'scale': 0.001}}
}


fiducial = {
    'params': {
        'ombh2': 0.022445,
        'omch2': 0.1205579307,
        'ns': 0.96,
        'As': 2.12605e-09,
        'tau': 0.05,
        'a0': -0.007589,
        'a1': 0.002008,
        'a2': -0.004127,
        'a3': 0.002918,
        'a4': -0.0006784,
        'H0': 67.0,
        'w': -1.0,
        'wa': 0.0,
        'mnu': 0.06,
        'b0_poly': 0.830703,
        'b1_poly': 1.190547,
        'b2_poly': -0.928357,
        'b3_poly': 0.423292
    }
}

In [ ]:
chi2 = {}  
for observable, path in data_path.items():

    print(f"\n--- Testing Observable: {observable} ---")
    chi2[observable] = {}
    info['likelihood']['LSS']['data_path'] = path
    
    
    for param, grid in param_grids.items():
        print(f"\n--- Iterating over Parameter: {param} ---")
        
        
        if param not in chi2[observable]:
            chi2[observable][param] = []
        
        info['params'] = deepcopy(fiducial['params'])
        info['params'][param] = deepcopy(open_params[param])
        
        
        for value in grid:

            
            info['sampler'] = {'evaluate': {'override': {param: value}}}
            
            try:
                update_info, sampler = run(info)  
                chi2_value = -2 * sampler.logposterior.loglike
                chi2[observable][param].append(chi2_value)

            except Exception as e:
                print(f"Error during sampling for {param}={value}, Observable={observable}: {e}")
                chi2[observable][param].append(None)

In [ ]:
import matplotlib.pyplot as plt

colors = ['darkorchid', 'goldenrod', 'forestgreen','crimson' ]
linestyles = ['-', '--', '-.', ':']


for param in param_grids.keys():
    plt.figure(figsize=(8, 6))
    
    
    for idx, (observable, results) in enumerate(chi2.items()):
        if param in results:  
            chi2_values = results[param]
            grid_values = param_grids[param]

            min_chi2 = min(chi2_values)
            min_index = chi2_values.index(min_chi2)
            min_param_value = grid_values[min_index]
            
            # Include the minimum chi2 in the legend label
            label = f'{observable} ($\chi_m^2$ = {min_chi2:.2f})'
            
            # Plot chi2 vs parameter grid values
            plt.plot(grid_values, chi2_values, color=colors[idx % len(colors)],
                     linestyle=linestyles[idx % len(linestyles)],
                     label=label, lw=2)
    
    
    plt.axvline(x=fiducial['params'][param], color='black', ls=':', label='Fiducial')
    
    
    plt.xlabel(f'${param}$', fontsize=14)
    plt.ylabel(r'$\chi^2$', fontsize=14)
    plt.title(f'$\chi^2$ vs ${param}$', fontsize=16)
    plt.legend(loc='best')
    
    
    plt.savefig('./plot/chi2_'+param+'.pdf')
    plt.tight_layout()
    plt.show()

